# Per-stage HF Datasets backup

Publishes the current state of `/Volumes/ds_work/alenj00/cap_cache/runs/full/` to a versioned
HuggingFace Dataset repo (`alenjani/cap-counterfactuals-stage-{a,b,c}`, all private). Called
between full-run stages by `scripts/cap_full_orchestrator.py` for off-cluster checkpointing.

Parameters:
- `stage`: one of `a`, `b`, `c`
- `source_dir`: Volume path of the run directory (default `/Volumes/.../runs/full/`)


In [ ]:
%pip install --upgrade huggingface_hub

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text('stage', 'a', 'Stage (a/b/c)')
dbutils.widgets.text('source_dir', '/Volumes/ds_work/alenj00/cap_cache/runs/full', 'Source Volume directory')
STAGE = dbutils.widgets.get('stage').lower()
SOURCE_DIR = dbutils.widgets.get('source_dir')
assert STAGE in ('a', 'b', 'c'), f'invalid stage: {STAGE}'
print(f'Stage: {STAGE}; source: {SOURCE_DIR}')

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, login

HF_WRITE_TOKEN = dbutils.secrets.get(scope='cap-secrets', key='hf-write-token')
login(token=HF_WRITE_TOKEN)

REPO_ID = f'alenjani/cap-counterfactuals-stage-{STAGE}'
RUN_DIR = Path(SOURCE_DIR)
PATH_IN_REPO = f'runs/full_stage_{STAGE}'

assert RUN_DIR.exists(), f'Source dir missing: {RUN_DIR}'
print(f'Source: {RUN_DIR}')
print(f'Target: {REPO_ID} -> /{PATH_IN_REPO}/')

In [ ]:
api = HfApi()
from huggingface_hub.errors import RepositoryNotFoundError
try:
    info = api.repo_info(repo_id=REPO_ID, repo_type='dataset')
    print(f'Dataset {REPO_ID} exists (private={info.private})')
except RepositoryNotFoundError:
    api.create_repo(repo_id=REPO_ID, repo_type='dataset', private=True, exist_ok=True)
    print(f'Created PRIVATE dataset: {REPO_ID}')

In [ ]:
# Per-stage README (compact). The canonical paper dataset card lives in the
# main `alenjani/cap-counterfactuals` repo at submission time.
STAGE_DESC = {
    'a': '200 IDs * 12 axes (skin x gender) * seed 42 = 2,400 images. Sufficient for Paper 1 ISR (single-seed).',
    'b': 'Cumulative through Stage A + adds age axis at seed 42 = 12,000 images. Unlocks Papers 2, 3 (single-seed).',
    'c': 'Cumulative through Stage B + seeds 137, 2718 across all cells = 36,000 images. Full instrument with 3-seed robustness.',
}
card = (
    f"---\n"
    f"license: cc-by-4.0\n"
    f"task_categories:\n"
    f"  - image-to-image\n"
    f"tags:\n"
    f"  - fairness\n"
    f"  - counterfactual\n"
    f"  - face-generation\n"
    f"  - audit\n"
    f"pretty_name: 'CAP - Stage {STAGE.upper()}'\n"
    f"---\n\n"
    f"# CAP - Stage {STAGE.upper()} snapshot\n\n"
    f"**Status:** Private off-cluster backup of CAP full-instrument generation through Stage {STAGE.upper()}.\n"
    f"NOT the canonical paper dataset (that will be `alenjani/cap-counterfactuals` at first paper submission).\n\n"
    f"## Stage {STAGE.upper()} contents\n\n"
    f"{STAGE_DESC[STAGE]}\n\n"
    f"See [the repo plans/003-full-instrument-plan.md](https://github.com/alenjani/counterfactual-audit-pipeline/blob/main/plans/003-full-instrument-plan.md) for the cumulative staging rationale.\n\n"
    f"## Generator settings\n\n"
    f"- FluxPuLIDNativeGenerator (architectural fix from Review 001 §4)\n"
    f"- FP8 quantization via optimum-quanto, Marlin disabled (memory-friendly mode)\n"
    f"- 768x768, 40 inference steps, id_weight=1.5, controlnet_conditioning_scale=0.6\n"
    f"- ControlNet pose conditioning, antelopev2 face encoder\n\n"
    f"## License\n\n"
    f"CC-BY-4.0 (data). FairFace seed images (CC-BY-4.0) used as identity anchors.\n"
    f"This dataset is private during research; flipped public on first paper acceptance.\n"
)
Path('/local_disk0/cap_dataset_README.md').write_text(card)
print(f'Card len: {len(card)}')

In [ ]:
import time
t0 = time.time()
api.upload_file(
    path_or_fileobj='/local_disk0/cap_dataset_README.md',
    path_in_repo='README.md',
    repo_id=REPO_ID,
    repo_type='dataset',
    commit_message=f'Stage {STAGE.upper()} card',
)
print(f'Card uploaded in {time.time() - t0:.1f}s')

In [ ]:
import time
t0 = time.time()
api.upload_folder(
    folder_path=str(RUN_DIR),
    path_in_repo=PATH_IN_REPO,
    repo_id=REPO_ID,
    repo_type='dataset',
    commit_message=f'Stage {STAGE.upper()} cumulative snapshot',
    ignore_patterns=['*.tmp', '*~', '*incremental_manifest*', '*.partial'],
)
wall = time.time() - t0
msg = f'Stage {STAGE.upper()} upload wall: {wall:.0f}s ({wall/60:.1f} min). https://huggingface.co/datasets/{REPO_ID}'
print(msg)
dbutils.notebook.exit(msg)